In [1]:
suppressPackageStartupMessages({
  library(readxl)
  library(dplyr)
  library(tidyr)
  library(ggplot2)
  library(car)
})

# Soil Sediment Geochemistry - GLMM Analysis Pipeline
### Notebook 01 - Data Audit & Assumption Checks

**Workbook:** `Soil_Final_Raw_Data.xlsx` (sheets `Carbon`, `Nitrogen`, `Phosphorus`).
**Responses selected (confirmed):** `soc`, `total_nitrogen`, `total_phos`.

This notebook audits the dataset before modelling: loading/tidying, integrity (missingness, duplicates, plot IDs),
Gamma-GLMM tractability (strict positivity, scale/CV), descriptives, normality (Shapiro-Wilk),
variance homogeneity (Brown-Forsythe / median-centred Levene), outlier screen (robust z),
and the repeated-measures design audit.

**Run order:** `01_data_audit.ipynb` -> `02_glmm_models.ipynb` -> `03_visualizations.ipynb`.


## 1 - Load and tidy the workbook

Treatment labels are kept exactly as recorded in each sheet (Carbon: `C`/`I`; Nitrogen and Phosphorus: `c`/`i`), per instruction (harmonise only in Carbon, untouched elsewhere). Within each sheet the control-type label (`C` or `c`) is set as the reference level. `period` is ordered `Week_0 < Week_3 < Week_6 < Week_9`.


In [2]:
path_xlsx <- "Soil_Final_Raw_Data.xlsx"
if (!file.exists(path_xlsx)) {
  alt <- file.path("Glmm Stats", path_xlsx)
  if (file.exists(alt)) path_xlsx <- normalizePath(alt)
}
stopifnot(file.exists(path_xlsx))
cat("Workbook:", path_xlsx, "\n\n")

cfg <- list(
  Carbon     = list(response = "soc"),
  Nitrogen   = list(response = "total_nitrogen"),
  Phosphorus = list(response = "total_phos")
)

tidy_sheet <- function(sheet, resp) {
  raw <- read_excel(path_xlsx, sheet = sheet)
  per_lv <- paste0("Week_", sort(unique(as.numeric(sub("Week_", "", raw$period)))))
  trt_u  <- unique(raw$treatment)
  trt_lv <- c(trt_u[toupper(trt_u) == "C"], trt_u[toupper(trt_u) == "I"])
  raw %>%
    mutate(
      nutrient      = sheet,
      response_name = resp,
      response      = as.numeric(.data[[resp]]),
      period_num    = as.numeric(sub("Week_", "", period)),
      period        = factor(period, levels = per_lv),
      treatment     = factor(treatment, levels = trt_lv),
      plot          = factor(plot),
      fin_dry_wt    = as.numeric(fin_dry_wt),
      dbd           = as.numeric(dbd)
    ) %>%
    select(any_of(c(
      "nutrient", "response_name", "response", "period", "period_num", "treatment", "plot",
      "pen_cm", "rec_cm", "vol", "con_wt", "wet_wt", "fin_wet_wt", "first_dry", "sec_dry", "fin_dry_wt", "dbd",
      "som", "soc", "car_den", "total_nitrogen", "nitrogen_den", "total_phos", "phos_den"
    )))
}

soil <- bind_rows(lapply(names(cfg), function(nm) tidy_sheet(nm, cfg[[nm]]$response)))

cat("Rows per nutrient:\n")
print(as.data.frame(soil %>% count(nutrient, response_name)))
cat("\nAnalysis column inventory (non-missing counts):\n")
print(as.data.frame(
  soil %>%
    summarise(across(c(period, treatment, plot, response), ~ sum(!is.na(.))))
))

Workbook: Soil_Final_Raw_Data.xlsx 



Rows per nutrient:


    nutrient  response_name  n
1     Carbon            soc 72
2   Nitrogen total_nitrogen 36
3 Phosphorus     total_phos 36



Analysis column inventory (non-missing counts):


  period treatment plot response
1    144       144  144      144


## 2 - integrity audit: duplicates, missingness, labels, plot IDs


In [3]:
dup_n <- soil %>% count(nutrient, period, treatment, plot) %>% filter(n > 1) %>% nrow()
cat(sprintf("Duplicate (nutrient x period x treatment x plot) keys: %d\n", dup_n))

miss_tab <- soil %>%
  summarise(across(everything(), ~ sum(is.na(.)))) %>%
  pivot_longer(everything(), names_to = "column", values_to = "n_missing") %>%
  filter(n_missing > 0)
if (nrow(miss_tab) == 0) {
  cat("Missing values: none in any column of the tidied analysis dataset.\n")
} else {
  print(as.data.frame(miss_tab))
}

cat("\nTreatment labels as recorded per sheet:\n")
print(as.data.frame(soil %>% distinct(nutrient, treatment) %>% arrange(nutrient)))

cat("\nResponse column mapping:\n")
print(as.data.frame(soil %>% distinct(nutrient, response_name)))

plot_sets <- soil %>%
  group_by(nutrient, period) %>%
  summarise(plot_set = paste(sort(unique(plot)), collapse = ","), .groups = "drop")
consist <- plot_sets %>% group_by(nutrient) %>% summarise(
  identical_plot_id_sets_across_periods = (n_distinct(plot_set) == 1),
  .groups = "drop"
)
print(as.data.frame(consist))

Duplicate (nutrient x period x treatment x plot) keys: 0


          column n_missing
1            som        72
2            soc        72
3        car_den        72
4 total_nitrogen       108
5   nitrogen_den       108
6     total_phos       108
7       phos_den       108



Treatment labels as recorded per sheet:


    nutrient treatment
1     Carbon         I
2     Carbon         C
3   Nitrogen         i
4   Nitrogen         c
5 Phosphorus         i
6 Phosphorus         c



Response column mapping:


    nutrient  response_name
1     Carbon            soc
2   Nitrogen total_nitrogen
3 Phosphorus     total_phos


    nutrient identical_plot_id_sets_across_periods
1     Carbon                                  TRUE
2   Nitrogen                                  TRUE
3 Phosphorus                                  TRUE


## 3 - Gamma-GLMM tractability: positivity & scale

A Gamma GLMM with a log link requires strictly **positive** responses. The CV (sd/mean) is scale-free:
under the Gamma family the variance is assumed proportional to the mean squared (CV constant).


In [4]:
skewness <- function(x) {
  n <- length(x); m <- mean(x); s <- sd(x)
  if (is.na(s) || s == 0) return(NA_real_)
  sqrt(n) * sum((x - m)^3) / (n - 1) / s^3
}
kurtosis_excess <- function(x) {
  n <- length(x); m <- mean(x); s <- sd(x)
  if (is.na(s) || s == 0 || n < 4) return(NA_real_)
  ((n + 1) * sum((x - m)^4) / (n - 1) / s^4 - 3 * (n - 1)^2 / ((n - 2) * (n - 3)))
}

scale_tab <- soil %>%
  group_by(nutrient, response_name) %>%
  summarise(
    n          = n(),
    min        = min(response),
    max        = max(response),
    mean       = mean(response),
    sd         = sd(response),
    cv         = sd(response) / mean(response),
    skew       = skewness(response),
    kurt_excess = kurtosis_excess(response),
    zeros      = sum(response == 0),
    negatives  = sum(response < 0),
    .groups    = "drop"
  )
print(as.data.frame(scale_tab %>% mutate(across(c(min, max, mean, sd, cv, skew, kurt_excess), ~ round(.x, 3)))))

n_zero     <- sum(soil$response == 0)
n_negative <- sum(soil$response < 0)
all_pos    <- (n_zero == 0) && (n_negative == 0) && all(is.finite(soil$response))
cat(sprintf("\nGamma GLMM positivity requirement (strictly positive responses): %s\n",
            ifelse(all_pos, "PASSED", "FAILED")))
cat(sprintf("Skewness sign check (right-skewed > 0 supports Gamma/log over normal LMM): %s\n",
            paste(sprintf("%s=%.2f", scale_tab$nutrient, scale_tab$skew), collapse = "; ")))

    nutrient  response_name  n    min    max   mean    sd    cv  skew
1     Carbon            soc 72  1.210  6.135  3.541 1.098 0.310 0.971
2   Nitrogen total_nitrogen 36  0.056  0.435  0.222 0.096 0.432 0.693
3 Phosphorus     total_phos 36 23.900 66.100 40.950 9.469 0.231 2.417
  kurt_excess zeros negatives
1     219.874     0         0
2      79.817     0         0
3     109.422     0         0



Gamma GLMM positivity requirement (strictly positive responses): PASSED


Skewness sign check (right-skewed > 0 supports Gamma/log over normal LMM): Carbon=0.97; Nitrogen=0.69; Phosphorus=2.42


## 4 - Descriptive statistics: response by treatment x period


In [5]:
desc_tab <- soil %>%
  group_by(nutrient, response_name, treatment, period) %>%
  summarise(
    n      = n(),
    mean   = mean(response),
    sd     = sd(response),
    median = median(response),
    min    = min(response),
    max    = max(response),
    .groups = "drop"
  )

for (nm in unique(desc_tab$nutrient)) {
  cat("\n===", nm, paste0("(response: ", unique(desc_tab$response_name[desc_tab$nutrient == nm]), ")"), "===\n")
  print(as.data.frame(
    desc_tab %>% filter(nutrient == nm) %>%
      mutate(across(c(mean, sd, median, min, max), ~ round(.x, 3)))
  ))
}


=== Carbon (response: soc) ===
  nutrient response_name treatment period n  mean    sd median   min   max
1   Carbon           soc         C Week_0 9 3.826 0.781  3.955 2.515 4.990
2   Carbon           soc         C Week_3 9 3.569 0.676  3.830 2.610 4.435
3   Carbon           soc         C Week_6 9 2.582 1.153  2.711 1.210 4.049
4   Carbon           soc         C Week_9 9 3.511 0.876  3.651 2.419 5.193
5   Carbon           soc         I Week_0 9 3.892 1.119  3.535 2.605 5.805
6   Carbon           soc         I Week_3 9 4.037 0.972  3.725 2.920 5.580
7   Carbon           soc         I Week_6 9 3.253 1.524  3.333 1.308 6.008
8   Carbon           soc         I Week_9 9 3.662 1.170  3.543 2.525 6.135

=== Nitrogen (response: total_nitrogen) ===
  nutrient  response_name treatment period n  mean    sd median   min   max
1 Nitrogen total_nitrogen         c Week_0 9 0.236 0.077  0.273 0.085 0.315
2 Nitrogen total_nitrogen         c Week_9 9 0.158 0.089  0.141 0.056 0.314
3 Nitrogen total_nit

## 5 - Assumption checks

**Note.** For a Gamma GLMM, normality of the *raw* response is not required; Shapiro-Wilk and skewness here
document how far the data depart from normal-theory ANOVA assumptions and support the Gamma/log choice.
Homoscedasticity is inspected via Brown-Forsythe (median-centred Levene); outliers via a robust-z screen.


In [6]:
shap_cell <- soil %>%
  group_by(nutrient, treatment, period) %>%
  summarise(
    n = n(),
    p_shapiro = if (n >= 3 && n <= 5000 && var(response) > 0) shapiro.test(response)$p.value else NA_real_,
    .groups = "drop"
  ) %>%
  mutate(normal_at_alpha05 = !is.na(p_shapiro) & p_shapiro >= 0.05) %>%
  ungroup()

print(as.data.frame(shap_cell %>% mutate(p_shapiro = signif(p_shapiro, 3))))

shap_verdict <- shap_cell %>%
  group_by(nutrient) %>%
  summarise(cells_tested = sum(!is.na(p_shapiro)),
            cells_non_normal = sum(!normal_at_alpha05, na.rm = TRUE),
            .groups = "drop")
cat("\nNormality verdict per nutrient (treatment x period cells):\n")
print(as.data.frame(shap_verdict))

overall_shapiro <- lapply(split(soil, soil$nutrient), function(d) shapiro.test(d$response)$p.value)
cat("\nPooled Shapiro-Wilk p-values (all rows per nutrient):\n")
for (i in seq_along(overall_shapiro)) {
  cat(sprintf("  %-10s p = %.4g\n", names(overall_shapiro)[i], overall_shapiro[[i]]))
}

     nutrient treatment period n p_shapiro normal_at_alpha05
1      Carbon         C Week_0 9     0.645              TRUE
2      Carbon         C Week_3 9     0.364              TRUE
3      Carbon         C Week_6 9     0.173              TRUE
4      Carbon         C Week_9 9     0.421              TRUE
5      Carbon         I Week_0 9     0.266              TRUE
6      Carbon         I Week_3 9     0.180              TRUE
7      Carbon         I Week_6 9     0.742              TRUE
8      Carbon         I Week_9 9     0.170              TRUE
9    Nitrogen         c Week_0 9     0.186              TRUE
10   Nitrogen         c Week_9 9     0.416              TRUE
11   Nitrogen         i Week_0 9     0.155              TRUE
12   Nitrogen         i Week_9 9     0.201              TRUE
13 Phosphorus         c Week_0 9     0.592              TRUE
14 Phosphorus         c Week_9 9     0.853              TRUE
15 Phosphorus         i Week_0 9     0.679              TRUE
16 Phosphorus         i 


Normality verdict per nutrient (treatment x period cells):


    nutrient cells_tested cells_non_normal
1     Carbon            8                0
2   Nitrogen            4                0
3 Phosphorus            4                0



Pooled Shapiro-Wilk p-values (all rows per nutrient):


  Carbon     p = 0.12
  Nitrogen   p = 0.212
  Phosphorus p = 0.8029


In [7]:
lev_tab <- bind_rows(lapply(split(soil, soil$nutrient), function(d) {
  grp <- interaction(d$treatment, d$period, sep = "_x_")
  lt  <- car::leveneTest(d$response, grp, center = "median")
  data.frame(
    nutrient = unique(d$nutrient),
    F_value  = as.numeric(lt[1, "F value"]),
    df1      = as.numeric(lt[1, "Df"]),
    df2      = as.numeric(lt[2, "Df"]),
    p_value  = as.numeric(lt[1, "Pr(>F)"])
  )
}))
print(as.data.frame(lev_tab %>% mutate(across(c(F_value, p_value), ~ signif(.x, 3)))))
homok <- all(lev_tab$p_value >= 0.05, na.rm = TRUE)
cat(sprintf("\nBrown-Forsythe variance-homogeneity verdict (all p >= 0.05): %s\n",
            ifelse(homok, "PASSED", "at least one nutrient shows variance heterogeneity")))

    nutrient F_value df1 df2 p_value
1     Carbon   0.903   7  64  0.5100
2   Nitrogen   0.166   3  32  0.9180
3 Phosphorus   3.470   3  32  0.0274



Brown-Forsythe variance-homogeneity verdict (all p >= 0.05): at least one nutrient shows variance heterogeneity


In [8]:
cellstats <- soil %>%
  group_by(nutrient, treatment, period) %>%
  summarise(med = median(response), madv = mad(response), .groups = "drop")

outj <- left_join(soil, cellstats, by = c("nutrient", "treatment", "period")) %>%
  mutate(robust_z = ifelse(is.finite(madv) & madv > 0, 0.6745 * (response - med) / madv, NA_real_))

flagged <- outj %>% filter(!is.na(robust_z) & abs(robust_z) > 3.5)
cat(sprintf("Outliers flagged (|robust z| > 3.5 within treatment x period cells): %d\n", nrow(flagged)))
if (nrow(flagged) > 0) {
  print(as.data.frame(flagged %>% select(nutrient, period, treatment, plot, response, robust_z)))
}

Outliers flagged (|robust z| > 3.5 within treatment x period cells): 3


  nutrient period treatment plot response  robust_z
1 Nitrogen Week_0         i    1    0.178 -5.042296
2 Nitrogen Week_0         i    2    0.148 -6.179656
3 Nitrogen Week_0         i    3    0.414  3.904936


## 6 - Repeated-measures design audit

The same `plot` IDs (1-9) appear in every sampling period within each nutrient sheet: if these are the same
physical plots resampled over time, `plot` acts as a random intercept factor (`(1 | plot)`), as fitted in Notebook 02.


In [9]:
design_tab <- soil %>%
  group_by(nutrient) %>%
  group_modify(~ {
    ct <- table(.x$treatment, droplevels(.x$period))
    ct <- ct[rowSums(ct) > 0, colSums(ct) > 0, drop = FALSE]
    data.frame(
      n_rows     = nrow(.x),
      n_plots    = n_distinct(.x$plot),
      periods    = paste(levels(droplevels(.x$period)), collapse = " < "),
      treatments = paste(levels(.x$treatment), collapse = " vs "),
      balanced   = all(ct == min(ct))
    )
  })
print(as.data.frame(design_tab))

reps <- soil %>%
  group_by(nutrient) %>%
  summarise(obs_per_plot_min = min(as.numeric(table(plot, droplevels(period)))),
            obs_per_plot_max = max(as.numeric(table(plot, droplevels(period)))),
            .groups = "drop")
print(as.data.frame(reps))

    nutrient n_rows n_plots                           periods       treatments
1     Carbon     72       9 Week_0 < Week_3 < Week_6 < Week_9 C vs I vs c vs i
2   Nitrogen     36       9                   Week_0 < Week_9 C vs I vs c vs i
3 Phosphorus     36       9                   Week_0 < Week_9 C vs I vs c vs i
  balanced
1     TRUE
2     TRUE
3     TRUE


    nutrient obs_per_plot_min obs_per_plot_max
1     Carbon                2                2
2   Nitrogen                2                2
3 Phosphorus                2                2


## 7 - Audit verdict


In [10]:
verdict_lines <- c(
  sprintf("1. Positivity (Gamma/log requirement): %s -> %s",
          ifelse(n_zero == 0 && n_negative == 0, "no zeros / no negatives", "violated"),
          ifelse(n_zero == 0 && n_negative == 0, "Gamma GLMM (log link) is valid", "consider a different family")),
  sprintf("2. Completeness: %d rows, %d missing values in the analysis columns, %d duplicate keys.",
          nrow(soil), sum(is.na(soil$response)), dup_n),
  sprintf("3. Normality (Shapiro-Wilk, treatment x period cells): %s.",
          paste(sprintf("%s %d/%d non-normal", shap_verdict$nutrient, shap_verdict$cells_non_normal, shap_verdict$cells_tested), collapse = "; ")),
  sprintf("4. Variance homogeneity (Brown-Forsythe per nutrient): %s.",
          ifelse(homok, "all p >= 0.05", "at least one p < 0.05")),
  sprintf("5. Outliers (robust z > 3.5): %d rows.", nrow(flagged)),
  sprintf("6. Design: %s.",
          paste(sprintf("%s (%s) balanced=%s, obs/plot=%d-%d", design_tab$nutrient, design_tab$periods, design_tab$balanced,
                        reps$obs_per_plot_min[match(design_tab$nutrient, reps$nutrient)],
                        reps$obs_per_plot_max[match(design_tab$nutrient, reps$nutrient)]), collapse = "; ")),
  "7. Modelling plan: response ~ treatment * period + (1|plot), Gamma(link=log); diagnostics via DHARMa.",
  "8. Near-normal raw responses do NOT invalidate the Gamma GLMM; they only permit a normal-LMM sensitivity check in Notebook 02."
)
cat(paste(verdict_lines, collapse = "\n"), "\n")

1. Positivity (Gamma/log requirement): no zeros / no negatives -> Gamma GLMM (log link) is valid
2. Completeness: 144 rows, 0 missing values in the analysis columns, 0 duplicate keys.
3. Normality (Shapiro-Wilk, treatment x period cells): Carbon 0/8 non-normal; Nitrogen 0/4 non-normal; Phosphorus 0/4 non-normal.
4. Variance homogeneity (Brown-Forsythe per nutrient): at least one p < 0.05.
5. Outliers (robust z > 3.5): 3 rows.
6. Design: Carbon (Week_0 < Week_3 < Week_6 < Week_9) balanced=TRUE, obs/plot=2-2; Nitrogen (Week_0 < Week_9) balanced=TRUE, obs/plot=2-2; Phosphorus (Week_0 < Week_9) balanced=TRUE, obs/plot=2-2.
7. Modelling plan: response ~ treatment * period + (1|plot), Gamma(link=log); diagnostics via DHARMa.
8. Near-normal raw responses do NOT invalidate the Gamma GLMM; they only permit a normal-LMM sensitivity check in Notebook 02. 


In [11]:
saveRDS(soil, "soil_tidied.rds")
write.csv(soil, "soil_tidied.csv", row.names = FALSE)
cat("Saved: soil_tidied.rds / soil_tidied.csv\n")
cat("\n--- Session info (appendix) ---\n")
sessionInfo()

Saved: soil_tidied.rds / soil_tidied.csv



--- Session info (appendix) ---


R version 4.6.0 (2026-04-24)
Platform: aarch64-apple-darwin23
Running under: macOS Tahoe 26.6.2

Matrix products: default
BLAS:   /Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.0.dylib 
LAPACK: /Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRlapack.dylib;  LAPACK version 3.12.1

locale:
[1] C.UTF-8/C.UTF-8/C.UTF-8/C/C.UTF-8/C.UTF-8

time zone: Asia/Manila
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] car_3.1-5     carData_3.0-6 ggplot2_4.0.3 tidyr_1.3.2   dplyr_1.2.1  
[6] readxl_1.4.5 

loaded via a namespace (and not attached):
 [1] gtable_0.3.6       jsonlite_2.0.0     compiler_4.6.0     crayon_1.5.3      
 [5] tidyselect_1.2.1   IRdisplay_1.1      systemfonts_1.3.2  scales_1.4.0      
 [9] textshaping_1.0.5  uuid_1.2-2         fastmap_1.2.0      IRkernel_1.3.2    
[13] R6_2.6.1           generics_0.1.4     Formula_1.2-6      tibble_3.3.1    